# 08 — Admin2 pretty Excel summaries

Produces color-coded, hazard-grouped Excel workbooks for **one country's admin2 units**, straight from the global admin2 CSVs (notebook 02 output). No aggregation — the class/index/exposure values are used as-is.

Two views per country:
- **`{country}_adm2_wide_hazard_exposure_{date}.xlsx`** — per hazard: Exposure Class | Exposure Index | Hazard Index.
- **`{country}_adm2_exposure_{date}.xlsx`** — front: Total / Child population; per hazard: Class (Total) | Class (Children) | Exposed (Total) | Exposed (Children).

Rows are grouped by admin1 (then worst class first within each). Status-flagged units are kept, with blank/uncolored class cells.

Styling, palette and labels come from `config/class_palette.json`, `config/column_labels.json`, and `hazard_info.json` `display_name`, via `pipeline.excel_report`.

In [ ]:
import sys
from pathlib import Path

SUBNATIONAL_ROOT = Path.cwd().parent
LIB_DIR = SUBNATIONAL_ROOT / "lib"
CONFIG_DIR = SUBNATIONAL_ROOT / "config"
for p in (str(SUBNATIONAL_ROOT), str(LIB_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import aggregation_functions as agg
from pipeline import excel_report as xr

pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters (edit these)
# ============================================================
COUNTRY_UCODE = "EGY_V1"
PROD_DATE = "2026-07-13"
SOURCE_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali\global\CSV"
OUTPUT_FOLDER = "output"

out_dir = f"{OUTPUT_FOLDER}/results_admin2/{COUNTRY_UCODE}"
import os
os.makedirs(out_dir, exist_ok=True)

cfg = xr.load_report_config(CONFIG_DIR)
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
valid_codes = list(hazard_info["code"])

# Discover hazard CSVs, keep only hazards still in hazard_info.json (drops retired ones).
discovered = agg.discover_hazard_codes(SOURCE_FOLDER, PROD_DATE)
codes = [c for c in valid_codes if c in discovered]
dropped = [c for c in discovered if c not in valid_codes]
if dropped:
    print(f"skipping {len(dropped)} hazard(s) not in hazard_info.json: {dropped}")
print(f"hazards: {len(codes)}")

In [ ]:
# ============================================================
# Load this country's admin2 rows across all hazards (long form)
# ============================================================
KEEP = ["hazard_code", "adm0_ucode", "adm1_name", "adm2_ucode", "adm2_name",
        "cexpc", "u18_cexpc", "cexpi", "cindex", "aexp", "u18_aexp",
        "pop_total", "u18_pop_total", "status"]

frames = []
for code in codes:
    df = agg.load_hazard_global(code, PROD_DATE, SOURCE_FOLDER)
    c = df[df["adm0_ucode"] == COUNTRY_UCODE].copy()
    if c.empty:
        print(f"  {code}: no rows for {COUNTRY_UCODE}")
        continue
    c["hazard_code"] = code
    frames.append(c[[k for k in KEEP if k in c.columns]])

adm2_long = pd.concat(frames, ignore_index=True)
print(f"admin2 long: {adm2_long.shape} | admin2 units: {adm2_long.adm2_ucode.nunique()}")

In [ ]:
# ============================================================
# Helper: pivot the admin2 long table into a hazard-grouped wide frame
# ============================================================
IDX = ["adm1_name", "adm2_ucode", "adm2_name"]

def build_wide(values):
    """Return (wide_df, group_spans) for the given per-hazard value columns,
    with each hazard's columns adjacent and in hazard_info order."""
    wide = adm2_long[IDX].drop_duplicates().set_index(IDX)
    for v in values:
        p = adm2_long.pivot_table(index=IDX, columns="hazard_code", values=v, aggfunc="first")
        for code in codes:
            if code in p.columns:
                wide[f"{code}__{v}"] = p[code]
    wide = wide.reset_index()
    spans = [(code, [f"{code}__{v}" for v in values if f"{code}__{v}" in wide.columns]) for code in codes]
    spans = [(c, cols) for c, cols in spans if cols]
    return wide, spans

def sort_grouped(wide, spans, class_suffix="cexpc"):
    """Order rows by adm2_ucode (pcode) so every workbook lines up row-for-row.

    COs read these side by side, so the admin2 sequence must be identical in
    both views and stable across production dates. The per-hazard class columns
    remain in the sheet, so a reader can re-sort on them in Excel for a
    worst-first view.
    """
    key = "adm2_ucode" if "adm2_ucode" in wide.columns else "adm1_name"
    return wide.sort_values(key).copy()

In [ ]:
# ============================================================
# VIEW 1 — hazard_exposure: per hazard Exposure Class | Exposure Index | Hazard Index
# ============================================================
VALUES1 = ["cexpc", "cexpi", "cindex"]
wide1, spans1 = build_wide(VALUES1)
wide1 = sort_grouped(wide1, spans1, class_suffix="cexpc")

FRONT1 = ["adm1_name", "adm2_ucode", "adm2_name"]
FRONT_LABELS1 = {"adm1_name": "Admin1", "adm2_ucode": "Admin2 Code", "adm2_name": "Admin2 Name"}
VAR_LABELS1 = {"cexpc": "Exposure Class", "cexpi": "Exposure Index", "cindex": "Hazard Index"}

res1 = xr.write_wide_excel(
    wide1, f"{out_dir}/{COUNTRY_UCODE}_adm2_wide_hazard_exposure_{PROD_DATE}.xlsx",
    cfg=cfg, front_cols=FRONT1, group_spans=spans1,
    class_value_suffixes={"cexpc"}, front_labels=FRONT_LABELS1,
    variable_labels=VAR_LABELS1, sheet_name="hazard_exposure",
)
print(res1)

In [ ]:
# ============================================================
# VIEW 2 — exposure: front Total/Child population; per hazard
#          Class (Total) | Class (Children) | Exposed (Total) | Exposed (Children)
# ============================================================
VALUES2 = ["cexpc", "u18_cexpc", "aexp", "u18_aexp"]
wide2, spans2 = build_wide(VALUES2)

# Full admin2 population = max of pop_total across hazards (unmasked total).
pop = (adm2_long.groupby("adm2_ucode", as_index=False)
       .agg(pop_total=("pop_total", "max"), u18_pop_total=("u18_pop_total", "max")))
wide2 = wide2.merge(pop, on="adm2_ucode", how="left")
wide2 = sort_grouped(wide2, spans2, class_suffix="cexpc")

FRONT2 = ["adm1_name", "adm2_ucode", "adm2_name", "pop_total", "u18_pop_total"]
FRONT_LABELS2 = {"adm1_name": "Admin1", "adm2_ucode": "Admin2 Code", "adm2_name": "Admin2 Name",
                 "pop_total": "Total Population", "u18_pop_total": "Child Population"}
VAR_LABELS2 = {"cexpc": "Class (Total)", "u18_cexpc": "Class (Children)",
               "aexp": "Exposed (Total)", "u18_aexp": "Exposed (Children)"}
THOUSANDS = ({"pop_total", "u18_pop_total"}
             | {f"{c}__aexp" for c, _ in spans2} | {f"{c}__u18_aexp" for c, _ in spans2})

res2 = xr.write_wide_excel(
    wide2, f"{out_dir}/{COUNTRY_UCODE}_adm2_exposure_{PROD_DATE}.xlsx",
    cfg=cfg, front_cols=FRONT2, group_spans=spans2,
    class_value_suffixes={"cexpc", "u18_cexpc"}, front_labels=FRONT_LABELS2,
    variable_labels=VAR_LABELS2, thousands_cols=THOUSANDS, sheet_name="exposure",
)
print(res2)